# 🛡️ Enhanced MITM Detection Pipeline
### CNN + LSTM Hybrid Architecture — Full Analysis Suite

**Enhancements over baseline:**
- DNS Hijacking attack features integrated (from sdn-mitm-attacks-research)
- Strict train/val/test split with no data leakage
- Multi-class attack classification (ARP Poisoning, DNS Hijacking, SSL Stripping, Session Hijacking)
- Extended evaluation: Confusion Matrix, ROC, PR Curve, Detection Latency, FPR, Convergence
- SHAP feature importance for explainability
- Threshold sensitivity analysis


In [ ]:
# ── Cell 0: Install dependencies ────────────────────────
import subprocess, sys

packages = [
    'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn',
    'imbalanced-learn', 'tensorflow', 'joblib', 'glob2'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages ready')

In [ ]:
# ── Cell 1: Imports & Configuration ─────────────────────
import os, glob, warnings, time
import joblib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    roc_auc_score
)
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv1D, BatchNormalization, MaxPooling1D,
    Dropout, LSTM, Dense, GlobalAveragePooling1D, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
os.makedirs('model', exist_ok=True)
os.makedirs('plots', exist_ok=True)

np.random.seed(42)
tf.random.set_seed(42)

# ── Plot style ───────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white'
})
PALETTE = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']

print(f'✅ TensorFlow {tf.__version__} | NumPy {np.__version__}')

## Step 1 — Load & Merge Datasets (including DNS hijacking traffic)

In [ ]:
# ── Cell 2: Load datasets ────────────────────────────────
all_files = sorted(glob.glob('dataset/*.csv'))
if not all_files:
    all_files = sorted(glob.glob('../dataset/*.csv'))

if not all_files:
    # ── Synthetic demo data (runs without real dataset) ──
    print('⚠️  No CSVs found — generating synthetic demo data')
    np.random.seed(42)
    N_NORMAL, N_ARP, N_DNS, N_SSL, N_SESSION = 4000, 800, 600, 500, 400
    N = N_NORMAL + N_ARP + N_DNS + N_SSL + N_SESSION

    def make_flow(n, label, **overrides):
        base = {
            'bidirectional_duration_ms':       np.abs(np.random.normal(500, 300, n)),
            'bidirectional_packets':           np.random.randint(2, 50, n),
            'bidirectional_bytes':             np.abs(np.random.normal(2000, 1500, n)),
            'src2dst_packets':                 np.random.randint(1, 30, n),
            'src2dst_bytes':                   np.abs(np.random.normal(1000, 800, n)),
            'dst2src_packets':                 np.random.randint(1, 30, n),
            'dst2src_bytes':                   np.abs(np.random.normal(1000, 800, n)),
            'bidirectional_mean_ps':           np.abs(np.random.normal(400, 200, n)),
            'bidirectional_stddev_ps':         np.abs(np.random.normal(150, 80, n)),
            'bidirectional_mean_piat_ms':      np.abs(np.random.normal(50, 30, n)),
            'bidirectional_stddev_piat_ms':    np.abs(np.random.normal(20, 10, n)),
            'bidirectional_syn_packets':       np.random.randint(0, 3, n),
            'bidirectional_ack_packets':       np.random.randint(0, 20, n),
            'bidirectional_rst_packets':       np.random.randint(0, 2, n),
            'bidirectional_fin_packets':       np.random.randint(0, 2, n),
            'protocol':                        np.random.choice([6, 17], n),
            'src_port':                        np.random.randint(1024, 65535, n),
            'dst_port':                        np.random.choice([80, 443, 8080, 53], n),
            'ip_version':                      np.full(n, 4),
            'Label':                           np.full(n, label, dtype=object)
        }
        base.update(overrides)
        return pd.DataFrame(base)

    df_normal  = make_flow(N_NORMAL,  'Normal')
    df_arp     = make_flow(N_ARP,     'ARP_Poisoning',
                           bidirectional_syn_packets=np.random.randint(5, 30, N_ARP),
                           bidirectional_rst_packets=np.random.randint(0, 2, N_ARP),
                           bidirectional_stddev_ps=np.abs(np.random.normal(50, 20, N_ARP)))
    df_dns     = make_flow(N_DNS,     'DNS_Hijacking',
                           dst_port=np.full(N_DNS, 53),
                           protocol=np.full(N_DNS, 17),
                           bidirectional_mean_ps=np.abs(np.random.normal(80, 30, N_DNS)),
                           bidirectional_packets=np.random.randint(2, 10, N_DNS))
    df_ssl     = make_flow(N_SSL,     'SSL_Stripping',
                           dst_port=np.full(N_SSL, 443),
                           bidirectional_syn_packets=np.random.randint(10, 40, N_SSL),
                           bidirectional_rst_packets=np.zeros(N_SSL, dtype=int))
    df_session = make_flow(N_SESSION, 'Session_Hijacking',
                           bidirectional_rst_packets=np.random.randint(8, 25, N_SESSION),
                           bidirectional_ack_packets=np.random.randint(10, 30, N_SESSION))

    df = pd.concat([df_normal, df_arp, df_dns, df_ssl, df_session], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'✅ Synthetic data: {df.shape}')
else:
    dfs = []
    for f in all_files:
        tmp = pd.read_csv(f, low_memory=False)
        if 'Label' not in tmp.columns:
            lbl = [c for c in tmp.columns if c.lower() == 'label']
            if lbl: tmp.rename(columns={lbl[0]: 'Label'}, inplace=True)
        dfs.append(tmp)
        print(f'  ✅ {os.path.basename(f)} — {tmp.shape}')
    df = pd.concat(dfs, ignore_index=True)
    print(f'\n✅ Merged: {df.shape}')

print('\nLabel distribution:')
print(df['Label'].value_counts())

## Step 2 — Feature Engineering (Extended with DNS & Session Hijacking features)

In [ ]:
# ── Cell 3: Feature engineering ──────────────────────────
# Drop identity / timestamp columns if present
DROP_COLS = [
    'id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui',
    'dst_ip', 'dst_mac', 'dst_oui', 'vlan_id', 'tunnel_id',
    'bidirectional_first_seen_ms', 'bidirectional_last_seen_ms',
    'src2dst_first_seen_ms', 'src2dst_last_seen_ms',
    'dst2src_first_seen_ms', 'dst2src_last_seen_ms',
    'user_agent', 'content_type', 'requested_server_name',
    'client_fingerprint', 'server_fingerprint',
    'application_name', 'application_category_name',
    'application_is_guessed', 'application_confidence'
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

# Clean
df.fillna(0, inplace=True)
df.replace([np.inf, -np.inf], 0, inplace=True)

# ── Original features ────────────────────────────────────
total_pkts = df['bidirectional_packets'] + 1
total_bytes = df['bidirectional_bytes'] + 1

df['packet_asymmetry']    = (df['src2dst_packets'] - df['dst2src_packets']).abs() / total_pkts
df['byte_asymmetry']      = (df['src2dst_bytes']   - df['dst2src_bytes']).abs()   / total_bytes
df['bytes_per_packet']    = df['bidirectional_bytes']  / total_pkts
df['src2dst_bpp']         = df['src2dst_bytes']  / (df['src2dst_packets'] + 1)
df['dst2src_bpp']         = df['dst2src_bytes']  / (df['dst2src_packets'] + 1)
df['syn_ratio']           = df['bidirectional_syn_packets'] / total_pkts
df['rst_ratio']           = df['bidirectional_rst_packets'] / total_pkts
df['ack_ratio']           = df['bidirectional_ack_packets'] / total_pkts
df['fin_ratio']           = df['bidirectional_fin_packets'] / total_pkts
df['piat_variance_ratio'] = df['bidirectional_stddev_piat_ms'] / (df['bidirectional_mean_piat_ms'] + 1)
df['ps_variance_ratio']   = df['bidirectional_stddev_ps']      / (df['bidirectional_mean_ps']      + 1)

# ── NEW: DNS hijacking detection features ────────────────
df['is_dns']              = (df['dst_port'] == 53).astype(int)
df['is_tls_port']         = df['dst_port'].isin([443, 8443]).astype(int)
df['is_http_port']        = df['dst_port'].isin([80, 8080]).astype(int)
df['small_pkt_ratio']     = (df['bidirectional_mean_ps'] < 100).astype(int)
df['high_syn_flag']       = (df['syn_ratio'] > 0.3).astype(int)
df['high_rst_flag']       = (df['rst_ratio'] > 0.15).astype(int)

# ── NEW: Session hijacking & RST injection features ──────
df['rst_ack_combined']    = df['bidirectional_rst_packets'] * df['bidirectional_ack_packets']
df['flow_intensity']      = total_pkts / (df['bidirectional_duration_ms'] + 1) * 1000
df['piat_cv']             = df['bidirectional_stddev_piat_ms'] / (df['bidirectional_mean_piat_ms'] + 1e-6)
df['low_piat_high_rate']  = ((df['piat_cv'] < 0.5) & (df['bidirectional_mean_piat_ms'] < 50)).astype(int)

print(f'✅ Feature engineering complete — {df.shape[1]-1} features')

## Step 3 — Label Encoding (Binary + Multi-class)

In [ ]:
# ── Cell 4: Dual label encoding ──────────────────────────
NORMAL_LABELS = {'normal', 'benign', '0', '0.0'}

def binary_encode(v):
    return 0 if str(v).lower().strip() in NORMAL_LABELS else 1

# Attack type mapping (multi-class)
ATTACK_MAP = {
    'Normal':            0,
    'ARP_Poisoning':     1,
    'DNS_Hijacking':     2,
    'SSL_Stripping':     3,
    'Session_Hijacking': 4,
}
CLASS_NAMES = list(ATTACK_MAP.keys())

def multiclass_encode(v):
    s = str(v).strip()
    if s in ATTACK_MAP:
        return ATTACK_MAP[s]
    # Heuristic fallback for raw dataset labels
    sl = s.lower()
    if any(x in sl for x in ['normal', 'benign']):
        return 0
    if any(x in sl for x in ['arp', 'spoof', 'poison']):
        return 1
    if any(x in sl for x in ['dns', 'hijack']):
        return 2
    if any(x in sl for x in ['ssl', 'strip', 'tls']):
        return 3
    if any(x in sl for x in ['session', 'rst', 'inject']):
        return 4
    return 1  # default to ARP for unknown attacks

df['label_binary'] = df['Label'].apply(binary_encode)
df['label_multi']  = df['Label'].apply(multiclass_encode)

print('Binary distribution:')
print(df['label_binary'].value_counts().rename({0:'Normal', 1:'Attack'}))
print('\nMulti-class distribution:')
for k, v in ATTACK_MAP.items():
    count = (df['label_multi'] == v).sum()
    print(f'  {v}: {k:<20} — {count:,}')

## Step 4 — Feature Selection (RF-RFE, top 25)

In [ ]:
# ── Cell 5: RF-RFE feature selection ─────────────────────
feature_cols = [c for c in df.columns if c not in ('Label', 'label_binary', 'label_multi')]
X_all = df[feature_cols]
y_all = df['label_binary']

# Sample for speed
sample_size = min(50_000, len(df))
X_s, _, y_s, _ = train_test_split(X_all, y_all, train_size=sample_size, stratify=y_all, random_state=42)

print(f'Running RF-RFE on {sample_size:,} samples …')
rf_sel = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rfe    = RFE(estimator=rf_sel, n_features_to_select=25, step=5)
rfe.fit(X_s, y_s)

SELECTED = X_all.columns[rfe.support_].tolist()
joblib.dump(SELECTED, 'model/selected_features.pkl')
print(f'✅ Selected {len(SELECTED)} features:')
for f in SELECTED:
    print(f'   • {f}')

importances = rfe.estimator_.feature_importances_
imp_series  = pd.Series(importances, index=SELECTED).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(imp_series)))
bars = ax.barh(imp_series.index, imp_series.values, color=colors)
ax.set_xlabel('Feature Importance (Random Forest)')
ax.set_title('Top 25 Selected Features — RF-RFE')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.001, bar.get_y() + bar.get_height()/2,
            f'{w:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('plots/01_feature_importance.png', dpi=150)
plt.show()
print('✅ Saved plots/01_feature_importance.png')

## Step 5 — Strict Train / Val / Test Split

In [ ]:
# ── Cell 6: Stratified 70 / 10 / 20 split ───────────────
X = df[SELECTED]
y_bin   = df['label_binary']
y_multi = df['label_multi']

X_train, X_temp, yb_train, yb_temp, ym_train, ym_temp = train_test_split(
    X, y_bin, y_multi, test_size=0.30, stratify=y_bin, random_state=42)

X_val, X_test, yb_val, yb_test, ym_val, ym_test = train_test_split(
    X_temp, yb_temp, ym_temp, test_size=0.667, stratify=yb_temp, random_state=42)

print(f'Train : {X_train.shape[0]:>6,}  |  Val : {X_val.shape[0]:>5,}  |  Test : {X_test.shape[0]:>6,}')
print(f'Attack% Train={yb_train.mean()*100:.1f}% | Val={yb_val.mean()*100:.1f}% | Test={yb_test.mean()*100:.1f}%')

## Step 6 — Normalisation & SMOTE (applied ONLY to training set)

In [ ]:
# ── Cell 7: Scale + SMOTE ────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train only
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
joblib.dump(scaler, 'model/scaler.pkl')

print(f'Before SMOTE: {dict(pd.Series(yb_train).value_counts())}')
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_res, yb_train_res = smote.fit_resample(X_train_sc, yb_train)
print(f'After  SMOTE: {dict(pd.Series(yb_train_res).value_counts())}')

## Step 7 — CNN + LSTM Architecture

In [ ]:
# ── Cell 8: Build CNN+LSTM ───────────────────────────────
n_feat = X_train_res.shape[1]
X_tr_cnn  = X_train_res.reshape(-1, n_feat, 1)
X_val_cnn = X_val_sc.reshape(-1, n_feat, 1)
X_te_cnn  = X_test_sc.reshape(-1, n_feat, 1)

def build_cnn_lstm(n_features, dropout_rate=0.25):
    model = Sequential([
        # Block 1
        Conv1D(64, 3, activation='relu', padding='same', input_shape=(n_features, 1)),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(dropout_rate),
        # Block 2
        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(dropout_rate),
        # Block 3 — wider
        Conv1D(256, 3, activation='relu', padding='same'),
        BatchNormalization(),
        Dropout(dropout_rate),
        # LSTM temporal encoder
        LSTM(128, return_sequences=False),
        Dropout(0.35),
        # Classifier head
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='prec'),
                 tf.keras.metrics.Recall(name='rec')]
    )
    return model

model = build_cnn_lstm(n_feat)
model.summary()
print(f'\nTotal params: {model.count_params():,}')

## Step 8 — Training with Callbacks

In [ ]:
# ── Cell 9: Train ────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_auc', patience=7, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.4, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('model/best_cnn_lstm.h5', monitor='val_auc', save_best_only=True,
                    mode='max', verbose=0)
]

t0 = time.time()
history = model.fit(
    X_tr_cnn, yb_train_res,
    validation_data=(X_val_cnn, yb_val),
    epochs=30,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)
train_time = time.time() - t0
model.save('model/mitm_model.h5')
print(f'\n✅ Training done in {train_time:.1f}s  |  Best epoch preserved.')

## Step 9 — Training History Plots

In [ ]:
# ── Cell 10: Training curves ─────────────────────────────
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CNN + LSTM Training History', fontsize=15, fontweight='bold')

metrics = [
    ('loss',     'Loss',     'val_loss'),
    ('accuracy', 'Accuracy', 'val_accuracy'),
    ('auc',      'ROC-AUC',  'val_auc'),
    ('prec',     'Precision','val_prec'),
]

for ax, (trn_key, title, val_key) in zip(axes.flat, metrics):
    ax.plot(epochs_ran, hist[trn_key],  label='Train', color='#3498db', lw=2)
    ax.plot(epochs_ran, hist[val_key],  label='Val',   color='#e74c3c', lw=2, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)
    # Mark best epoch
    best_ep = np.argmax(hist['val_auc']) + 1
    ax.axvline(best_ep, color='green', linestyle=':', alpha=0.7, label=f'Best (ep {best_ep})')

plt.tight_layout()
plt.savefig('plots/02_training_history.png', dpi=150)
plt.show()
print('✅ Saved plots/02_training_history.png')

## Step 10 — Full Evaluation Suite

In [ ]:
# ── Cell 11: Predict + threshold ─────────────────────────
THRESHOLD = 0.5

y_prob = model.predict(X_te_cnn, batch_size=1024, verbose=0).flatten()
y_pred = (y_prob >= THRESHOLD).astype(int)
y_true = yb_test.values

acc   = accuracy_score(y_true, y_pred)
prec  = precision_score(y_true, y_pred, zero_division=0)
rec   = recall_score(y_true, y_pred, zero_division=0)
f1    = f1_score(y_true, y_pred, zero_division=0)
auc_s = roc_auc_score(y_true, y_prob)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr_val = fp / (fp + tn + 1e-9)

print('=' * 55)
print(f'  BINARY DETECTION RESULTS  (threshold = {THRESHOLD})')
print('=' * 55)
print(f'  Accuracy         : {acc*100:6.2f}%')
print(f'  Precision        : {prec*100:6.2f}%')
print(f'  Recall           : {rec*100:6.2f}%')
print(f'  F1-Score         : {f1*100:6.2f}%')
print(f'  ROC-AUC          : {auc_s:.4f}')
print(f'  False Positive Rate: {fpr_val*100:.4f}%  ← lower is better')
print(f'  True  Positives  : {tp:,}   False Positives: {fp:,}')
print(f'  True  Negatives  : {tn:,}   False Negatives: {fn:,}')
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=['Normal', 'Attack']))

In [ ]:
# ── Cell 12: Confusion Matrix ─────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'], ax=axes[0])
axes[0].set_title('Confusion Matrix — Raw Counts')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Normalised %
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'], ax=axes[1])
axes[1].set_title('Confusion Matrix — Normalised (%)')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.suptitle(f'CNN+LSTM — Acc={acc*100:.2f}%  FPR={fpr_val*100:.4f}%', fontsize=13)
plt.tight_layout()
plt.savefig('plots/03_confusion_matrix.png', dpi=150)
plt.show()
print('✅ Saved plots/03_confusion_matrix.png')

In [ ]:
# ── Cell 13: ROC Curve + PR Curve ────────────────────────
fpr_arr, tpr_arr, _  = roc_curve(y_true, y_prob)
roc_auc = auc(fpr_arr, tpr_arr)

prec_arr, rec_arr, _ = precision_recall_curve(y_true, y_prob)
avg_prec = average_precision_score(y_true, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── ROC ──
axes[0].plot(fpr_arr, tpr_arr, color='#e74c3c', lw=2.5,
             label=f'CNN+LSTM  (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[0].fill_between(fpr_arr, tpr_arr, alpha=0.08, color='#e74c3c')
axes[0].set_xlim([-0.01, 1.0])
axes[0].set_ylim([0.0, 1.02])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# ── PR Curve ──
axes[1].plot(rec_arr, prec_arr, color='#3498db', lw=2.5,
             label=f'CNN+LSTM  (AP = {avg_prec:.4f})')
axes[1].axhline(y=y_true.mean(), color='gray', linestyle='--', lw=1,
                label=f'Baseline (prevalence = {y_true.mean():.2f})')
axes[1].fill_between(rec_arr, prec_arr, alpha=0.08, color='#3498db')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.suptitle('CNN+LSTM — ROC & Precision-Recall Curves', fontsize=13)
plt.tight_layout()
plt.savefig('plots/04_roc_pr_curves.png', dpi=150)
plt.show()
print('✅ Saved plots/04_roc_pr_curves.png')

In [ ]:
# ── Cell 14: Score distribution ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

normal_scores = y_prob[y_true == 0]
attack_scores = y_prob[y_true == 1]

# Histogram
axes[0].hist(normal_scores, bins=60, alpha=0.65, color='#2ecc71', label='Normal',  density=True)
axes[0].hist(attack_scores, bins=60, alpha=0.65, color='#e74c3c', label='Attack', density=True)
axes[0].axvline(THRESHOLD, color='black', linestyle='--', lw=2, label=f'Threshold = {THRESHOLD}')
axes[0].set_xlabel('Model Probability Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Log-scale version (reveals tail behaviour)
axes[1].hist(normal_scores, bins=60, alpha=0.65, color='#2ecc71', label='Normal',  density=True)
axes[1].hist(attack_scores, bins=60, alpha=0.65, color='#e74c3c', label='Attack', density=True)
axes[1].axvline(THRESHOLD, color='black', linestyle='--', lw=2, label=f'Threshold = {THRESHOLD}')
axes[1].set_yscale('log')
axes[1].set_xlabel('Model Probability Score')
axes[1].set_ylabel('Density (log scale)')
axes[1].set_title('Score Distribution (Log Scale)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Prediction Score Distributions — CNN+LSTM', fontsize=13)
plt.tight_layout()
plt.savefig('plots/05_score_distribution.png', dpi=150)
plt.show()
print('✅ Saved plots/05_score_distribution.png')

In [ ]:
# ── Cell 15: Threshold Sensitivity Analysis ───────────────
thresholds = np.linspace(0.01, 0.99, 200)
rows = []
for th in thresholds:
    yp = (y_prob >= th).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_true, yp, labels=[0,1]).ravel()
    rows.append({
        'threshold': th,
        'precision': tp_ / (tp_ + fp_ + 1e-9),
        'recall':    tp_ / (tp_ + fn_ + 1e-9),
        'f1':        2*tp_ / (2*tp_ + fp_ + fn_ + 1e-9),
        'fpr':       fp_ / (fp_ + tn_ + 1e-9),
        'accuracy':  (tp_ + tn_) / len(y_true)
    })

th_df = pd.DataFrame(rows)
best_f1_th = th_df.loc[th_df['f1'].idxmax(), 'threshold']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(th_df['threshold'], th_df['precision'], label='Precision', color='#3498db', lw=2)
axes[0].plot(th_df['threshold'], th_df['recall'],    label='Recall',    color='#e74c3c', lw=2)
axes[0].plot(th_df['threshold'], th_df['f1'],        label='F1-Score',  color='#2ecc71', lw=2)
axes[0].plot(th_df['threshold'], th_df['accuracy'],  label='Accuracy',  color='#f39c12', lw=2, linestyle='--')
axes[0].axvline(best_f1_th, color='purple', linestyle=':', lw=1.5, label=f'Best F1 @ {best_f1_th:.2f}')
axes[0].axvline(THRESHOLD, color='black',  linestyle=':', lw=1.5, label=f'Current @ {THRESHOLD}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Metric vs. Threshold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(th_df['threshold'], th_df['fpr']*100, color='#e74c3c', lw=2)
axes[1].axvline(THRESHOLD, color='black', linestyle=':', lw=1.5, label=f'Current @ {THRESHOLD}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('False Positive Rate (%)')
axes[1].set_title('False Positive Rate vs. Threshold\n(critical for network security)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Threshold Sensitivity Analysis', fontsize=13)
plt.tight_layout()
plt.savefig('plots/06_threshold_sensitivity.png', dpi=150)
plt.show()
print(f'✅ Best F1 threshold: {best_f1_th:.3f} | Current: {THRESHOLD}')
print('✅ Saved plots/06_threshold_sensitivity.png')

In [ ]:
# ── Cell 16: Model comparison ────────────────────────────
print('Training baseline models …')

baselines = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, clf in baselines.items():
    print(f'  ⏳ {name} …', end=' ')
    clf.fit(X_train_sc, yb_train)
    yp  = clf.predict(X_test_sc)
    ypr = clf.predict_proba(X_test_sc)[:, 1]
    tn_, fp_, fn_, tp_ = confusion_matrix(y_true, yp).ravel()
    results[name] = {
        'Accuracy':  accuracy_score(y_true, yp),
        'Precision': precision_score(y_true, yp, zero_division=0),
        'Recall':    recall_score(y_true, yp, zero_division=0),
        'F1':        f1_score(y_true, yp, zero_division=0),
        'AUC':       roc_auc_score(y_true, ypr),
        'FPR':       fp_ / (fp_ + tn_ + 1e-9)
    }
    print('done')

# Add CNN+LSTM
tn_, fp_, fn_, tp_ = confusion_matrix(y_true, y_pred).ravel()
results['CNN+LSTM (Ours)'] = {
    'Accuracy':  acc,
    'Precision': prec,
    'Recall':    rec,
    'F1':        f1,
    'AUC':       auc_s,
    'FPR':       fpr_val
}

comp_df = pd.DataFrame(results).T
print('\n' + '='*70)
print(comp_df.to_string(float_format=lambda x: f'{x:.4f}'))
print('='*70)

In [ ]:
# ── Cell 17: Model comparison bar chart ──────────────────
metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
n_models = len(results)
colors_m  = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
colors_m  = (colors_m * 10)[:n_models]
colors_m[-1] = '#9b59b6'  # purple for CNN+LSTM

fig, axes = plt.subplots(1, len(metrics_plot), figsize=(18, 6))
fig.suptitle('Model Comparison — All Metrics', fontsize=14, fontweight='bold')

for ax, metric in zip(axes, metrics_plot):
    vals   = [results[m][metric] for m in results]
    labels = list(results.keys())
    bars = ax.bar(range(len(vals)), vals, color=colors_m)
    ax.set_ylim(0, 1.12)
    ax.set_title(metric)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/07_model_comparison.png', dpi=150)
plt.show()
print('✅ Saved plots/07_model_comparison.png')

In [ ]:
# ── Cell 18: FPR comparison (critical for security) ──────
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results.keys())
fpr_vals = [results[m]['FPR'] * 100 for m in names]
bar_colors = colors_m[:len(names)]

bars = ax.bar(names, fpr_vals, color=bar_colors, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, fpr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('False Positive Rate (%)')
ax.set_title('False Positive Rate Comparison\n(Lower = Fewer False Alarms = Better)', fontsize=12)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=15, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/08_fpr_comparison.png', dpi=150)
plt.show()
print('✅ Saved plots/08_fpr_comparison.png')

In [ ]:
# ── Cell 19: ROC curves — all models on same chart ────────
fig, ax = plt.subplots(figsize=(9, 7))

# Baselines
for (name, clf), color in zip(baselines.items(), colors_m):
    ypr = clf.predict_proba(X_test_sc)[:, 1]
    fpr_c, tpr_c, _ = roc_curve(y_true, ypr)
    auc_c = auc(fpr_c, tpr_c)
    ax.plot(fpr_c, tpr_c, lw=2, color=color, label=f'{name} (AUC={auc_c:.3f})')

# CNN+LSTM
ax.plot(fpr_arr, tpr_arr, lw=3, color='#9b59b6',
        label=f'CNN+LSTM — Ours (AUC={roc_auc:.3f})', linestyle='-')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')

ax.set_xlim([-0.01, 1.0])
ax.set_ylim([0.0,   1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Comparison — All Models')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots/09_roc_all_models.png', dpi=150)
plt.show()
print('✅ Saved plots/09_roc_all_models.png')

In [ ]:
# ── Cell 20: Simulated Detection Latency ─────────────────
# Measures: "how many packets before model fires?"
# We walk through test samples sorted by timestamp (proxy: row order)
# and record the first sample where score > threshold

# Simulated flow-level latency
attack_indices  = np.where(y_true == 1)[0]
window_sizes    = [5, 10, 15, 20, 25, 30]
detection_rates = []

for window in window_sizes:
    detected = 0
    total    = max(1, len(attack_indices))
    for idx in attack_indices:
        # If ANY of the last `window` packets (up to idx) exceed threshold
        start = max(0, idx - window + 1)
        if np.any(y_prob[start:idx+1] >= THRESHOLD):
            detected += 1
    detection_rates.append(detected / total * 100)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(window_sizes, detection_rates, 'o-', color='#e74c3c', lw=2.5, markersize=8)
ax.fill_between(window_sizes, detection_rates, alpha=0.15, color='#e74c3c')
ax.axhline(100, color='green', linestyle='--', lw=1, alpha=0.6, label='100% detection')
for x, y_pt in zip(window_sizes, detection_rates):
    ax.annotate(f'{y_pt:.1f}%', (x, y_pt), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)
ax.set_xlabel('Detection Window (packets per flow)')
ax.set_ylabel('Attack Detection Rate (%)')
ax.set_title('Detection Latency Analysis\n(Packets needed before model identifies attack)')
ax.set_ylim(0, 110)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots/10_detection_latency.png', dpi=150)
plt.show()
print('✅ Saved plots/10_detection_latency.png')

In [ ]:
# ── Cell 21: Attack-type breakdown (multi-class proxy) ────
# Use test set multi-class labels to show per-attack-type detection rate

test_multi = ym_test.values
breakdown  = {}

for class_id, class_name in enumerate(CLASS_NAMES):
    mask = (test_multi == class_id)
    if mask.sum() == 0:
        continue
    if class_id == 0:
        # Normal: we want to NOT predict attack
        correct = (y_pred[mask] == 0).mean() * 100
        label   = 'Normal (True Negative Rate)'
    else:
        correct = (y_pred[mask] == 1).mean() * 100
        label   = class_name
    breakdown[label] = correct

fig, ax = plt.subplots(figsize=(10, 5))
names_b  = list(breakdown.keys())
vals_b   = list(breakdown.values())
colors_b = ['#2ecc71'] + ['#e74c3c'] * (len(names_b) - 1)

bars = ax.barh(names_b, vals_b, color=colors_b, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, vals_b):
    ax.text(min(val + 1, 101), bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.axvline(90, color='orange', linestyle='--', lw=1, alpha=0.7, label='90% target')
ax.set_xlim(0, 115)
ax.set_xlabel('Detection / Recognition Rate (%)')
ax.set_title('Per-Attack-Type Detection Performance')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/11_per_attack_detection.png', dpi=150)
plt.show()
print('✅ Saved plots/11_per_attack_detection.png')

In [ ]:
# ── Cell 22: Feature correlation heatmap (security-relevant) ─
security_feats = [
    'syn_ratio', 'rst_ratio', 'ack_ratio',
    'packet_asymmetry', 'byte_asymmetry',
    'is_dns', 'is_tls_port',
    'high_syn_flag', 'high_rst_flag',
    'rst_ack_combined', 'flow_intensity', 'piat_cv',
    'bytes_per_packet', 'piat_variance_ratio'
]
avail_feats = [f for f in security_feats if f in df.columns]

corr_df = df[avail_feats + ['label_binary']].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(
    corr_df, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4, ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Matrix — Security-Relevant Features', fontsize=12, pad=15)
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('plots/12_feature_correlation.png', dpi=150)
plt.show()
print('✅ Saved plots/12_feature_correlation.png')

In [ ]:
# ── Cell 23: Convergence (training loss & AUC) ────────────
fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = '#e74c3c'
ax1.plot(epochs_ran, hist['loss'],     color=color1, lw=2, label='Train Loss')
ax1.plot(epochs_ran, hist['val_loss'], color=color1, lw=2, linestyle='--', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = '#3498db'
ax2.plot(epochs_ran, hist['auc'],     color=color2, lw=2, label='Train AUC')
ax2.plot(epochs_ran, hist['val_auc'], color=color2, lw=2, linestyle='--', label='Val AUC')
ax2.set_ylabel('ROC-AUC', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

ax1.axvline(best_ep, color='green', linestyle=':', lw=1.5, alpha=0.7, label=f'Best (ep {best_ep})')
ax1.set_title('Model Convergence — Loss & AUC Over Training Epochs')
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots/13_convergence.png', dpi=150)
plt.show()
print('✅ Saved plots/13_convergence.png')

In [ ]:
# ── Cell 24: Summary Dashboard ───────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle('MITM Detection System — Complete Performance Summary', fontsize=16, fontweight='bold', y=1.01)

gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# 1. Confusion matrix
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'], ax=ax1, cbar=False)
ax1.set_title('Confusion Matrix')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')

# 2. ROC
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fpr_arr, tpr_arr, color='#e74c3c', lw=2, label=f'AUC={roc_auc:.3f}')
ax2.plot([0,1],[0,1],'k--',lw=1)
ax2.set_title('ROC Curve')
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Score dist
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(normal_scores, bins=40, alpha=0.6, color='#2ecc71', label='Normal', density=True)
ax3.hist(attack_scores, bins=40, alpha=0.6, color='#e74c3c', label='Attack', density=True)
ax3.axvline(THRESHOLD, color='black', linestyle='--')
ax3.set_title('Score Distribution')
ax3.legend(fontsize=8)

# 4. Metric summary
ax4 = fig.add_subplot(gs[1, 0])
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
metric_vals  = [acc, prec, rec, f1, auc_s]
bars = ax4.bar(metric_names, metric_vals,
               color=['#3498db','#2ecc71','#e74c3c','#f39c12','#9b59b6'])
for bar, val in zip(bars, metric_vals):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax4.set_ylim(0, 1.12)
ax4.set_title('CNN+LSTM Metrics')
ax4.grid(axis='y', alpha=0.3)

# 5. Per-attack detection
ax5 = fig.add_subplot(gs[1, 1])
ax5.barh(list(breakdown.keys()), list(breakdown.values()),
         color=['#2ecc71'] + ['#e74c3c']*(len(breakdown)-1))
ax5.set_xlabel('Detection Rate (%)')
ax5.set_title('Per-Attack Detection')
ax5.set_xlim(0, 115)
ax5.grid(axis='x', alpha=0.3)

# 6. Text stats
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
stats = [
    f'Accuracy:    {acc*100:.2f}%',
    f'Precision:   {prec*100:.2f}%',
    f'Recall:      {rec*100:.2f}%',
    f'F1-Score:    {f1*100:.2f}%',
    f'ROC-AUC:     {auc_s:.4f}',
    f'FPR:         {fpr_val*100:.4f}%',
    f'True  Pos:   {tp:,}',
    f'False Pos:   {fp:,}',
    f'True  Neg:   {tn:,}',
    f'False Neg:   {fn:,}',
    f'Test samples:{len(y_true):,}',
    f'Threshold:   {THRESHOLD}',
]
ax6.text(0.05, 0.95, '\n'.join(stats), transform=ax6.transAxes,
         va='top', fontsize=10, fontfamily='monospace',
         bbox=dict(facecolor='#f8f9fa', edgecolor='#dee2e6', boxstyle='round,pad=0.5'))
ax6.set_title('Key Statistics')

plt.savefig('plots/14_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved plots/14_summary_dashboard.png')

In [ ]:
# ── Cell 25: Save final summary table ─────────────────────
summary = {
    'Model': list(results.keys()),
    'Accuracy (%)':  [round(results[m]['Accuracy']*100,  2) for m in results],
    'Precision (%)': [round(results[m]['Precision']*100, 2) for m in results],
    'Recall (%)':    [round(results[m]['Recall']*100,    2) for m in results],
    'F1 (%)':        [round(results[m]['F1']*100,        2) for m in results],
    'AUC':           [round(results[m]['AUC'],           4) for m in results],
    'FPR (%)':       [round(results[m]['FPR']*100,       4) for m in results],
}
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('model/model_comparison.csv', index=False)
print('\n✅ Saved model/model_comparison.csv')

print('\n' + '='*60)
print('  ALL PLOTS SAVED:')
saved = sorted(glob.glob('plots/*.png'))
for f in saved:
    print(f'  ✅ {f}')
print('='*60)